# Corpus

What this notebook establishes, before any question about safety or readability
is asked of the corpus: how many requests were sent, how many came back, and how
many of those can carry a safety judgement. Those counts are the denominators of
every safety rate in the thesis, and a rate whose denominator is not stated is
not a result. The readability denominator is a different question with its own
missingness, and it belongs with the measures, so it is asked in
`16_readability` rather than here.

This notebook depends on `scripts/` and on the calibration written by
`11_annotation`, which sets which annotated characteristics may carry a test.
Those coefficients are reported in Section~\ref{subsec:judge} of the experiments
chapter and are not repeated here: this notebook only checks that the decision
they produced still matches what `scripts/analysis.py` declares.

Five sections, in the order the corpus itself moves through:

| | Section | Question |
|---|---|---|
| C.2 | Integrity | do the properties the analysis assumes actually hold |
| C.3 | Flow and denominators | submitted, blocked, returned, classified |
| C.4 | Provider blocking | is a provider-blocked request patterned on the stated age |
| C.5 | Replicate stability | how much of an outcome survives a resample at the same setting |

**Conventions used here and in the three analysis notebooks that follow.**

A provider block is a **platform outcome**, not a model refusal. The request was
provider-blocked with no response text returned, so it is neither a Refusal nor a
Compliance and it is not sent to the classifier. It is analysed in C.4 in its
own right and excluded from the safety denominator everywhere else.

The **primary denominator is returned responses**. The end-to-end view, which
counts a blocked request as content that did not reach the user, is a secondary
sensitivity analysis and is
run in the safety notebook rather than here.

The **scenario is the inferential unit**. Every contrast is paired within a
scenario and every interval resamples scenarios. The three replicates are
averaged to the scenario x condition cell before any test, and are used in C.5
to quantify stochastic stability rather than to enlarge the sample.

Statistics, paths and the test register come from `scripts/analysis.py`. This
notebook registers into the `platform blocking` family, secondary. It is not
added to the primary hypothesis family,
which follows from the research questions and was settled before the final
inferential analysis, and is not reopened here.


## C.1 Setup


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import analysis
from analysis import (CELLS, CELL_ORDER, CONDITION_LABEL, DELIVERY, LABEL_ORDER,
                      SIGNAL, TRAJECTORY, present,
                      IMPLICIT, IMPLICIT_MINOR, MACRO, NAME, NEUTRAL, ORDER,
                      RUBRIC, STATED_ADULT, STATED_MINOR,
                      Register, bootstrap_paired, bootstrap_rate, contrast,
                      derive, differences, interval, leave_one_out,
                      permutation_paired, publish, pvalue, read_corpus,
                      save_figure, write_captions)
from settings import ANNOTATION_DIR, SAFETY, measure_column

mpl.rcParams.update(analysis.STYLE)
pd.set_option('display.width', 200, 'display.max_columns', 40,
              'display.max_colwidth', 64)

REGISTER = Register('corpus')


In [2]:
# The corpus, the outcome cells and the numeric indicators are defined once in
# scripts/analysis.py and imported. read_corpus leaves every field as the
# classifier wrote it, so C.2 can assert on the raw values; derive is applied
# after those assertions pass.
raw = read_corpus()
JUDGEMENT_SOURCE = raw.attrs['source']

print(f'{len(raw):,} judgements from {JUDGEMENT_SOURCE}')


46,800 judgements from 6 files in classification/


## C.2 Integrity

The properties everything downstream assumes are asserted here, before anything
is computed. A failure stops the notebook rather than producing a number that
looks reasonable.

These are properties of a well formed corpus, not statements about what the
corpus contains. How many requests a provider blocked is a result, so it is
reported rather than asserted: asserting it would make the notebook halt unless
it reproduced a number it is meant to be measuring. What `analysis.check_corpus`
does assert is the invariant that makes the count meaningful, namely that every
request is either returned or blocked and never both or neither.

Two of the checks here are worth naming. A blocked request was labelled from the
raw provider record without a classifier call, so it must carry no rubric fields;
if it carried any, a verdict was invented about a reply that does not exist. And
every returned response must fall in exactly one of the four outcome cells, since
the cell is what every safety rate is built from.


In [3]:
EMPTY = {'', 'nan', 'none', 'null', 'false', '0', '<na>'}


# Define function to say whether a column holds anything, treating the several
# ways a provider writes an absent value as absent
def carries(column):
    return ~column.fillna('').astype(str).str.strip().str.lower().isin(EMPTY)


blocked_rows = raw['answer'] == analysis.BLOCKED
returned_rows = raw['responded']

checks = {
    'no request failed with an error':
        not bool(carries(raw['error']).any()),
    'the primary outcome parsed on every returned response':
        not bool(raw.loc[returned_rows, 'unreadable'].fillna('').str.contains(
            'delivery response|answer', case=False, regex=True).any()),
    'a blocked request carries no rubric field':
        not any(bool(carries(raw.loc[blocked_rows, column]).any())
                for column in RUBRIC),
    # Which characteristics may carry a test comes out of the calibration in
    # notebook 11. Nothing is recomputed here; the decision is checked against
    # what scripts/analysis.py declares, so a recalibration cannot quietly
    # resize a declared family.
    'the agreement floor admits the declared characteristics':
        analysis.reliable_fields() >= set(analysis.TESTABLE),
    'every returned response falls in one outcome cell':
        int(sum(CELLS.get((answer, delivery)) is None for answer, delivery
                in zip(raw.loc[returned_rows, 'answer'],
                       raw.loc[returned_rows, DELIVERY]))) == 0,
}

failed = [name for name, passed in checks.items() if not passed]
assert not failed, 'corpus assertion failed: ' + '; '.join(failed)

frame = derive(raw)
frame.attrs['fingerprint'] = analysis.check_corpus(frame)
returned_rows = frame['responded']

# Two departures from the ideal corpus, counted rather than asserted. Neither
# touches the primary outcome: a characteristic that did not parse is missing
# rather than absent, and an alternative marked on a delivered compliance is
# ignored once the request was delivered.
unparsed = int(carries(raw.loc[returned_rows, 'unreadable']).sum())
anomaly = int((returned_rows & raw['answer'].eq('Compliance')
               & raw[DELIVERY].eq('Yes')
               & raw['alternative_response'].eq('Yes')).sum())

print(f'{analysis.fingerprint_line(frame)}')
print(f'{len(checks)} integrity checks passed; {unparsed} unparsed rubric '
      f'fields and {anomaly} alternatives on a delivered compliance, both '
      f'treated as missing')


46,800 requests, 160 blocked, 46,640 returned, 200 scenarios x 13 conditions x 6 models, rubric e5f836fffbf6
5 integrity checks passed; 0 unparsed rubric fields and 13 alternatives on a delivered compliance, both treated as missing


## C.3 Corpus flow and denominators

Every request follows the same path, and each step removes rows for a different
reason. Submitted is the design. Blocked is a provider decision taken before any
response text was returned. Returned is what the model produced, and is the
denominator of every response-conditional safety rate. It is not the denominator
of the block rate, which counts requests that never returned, nor of the
end-to-end sensitivity analysis in Chapter 5.2, both of which are computed over
submitted requests.

The path does not continue in the same way for the readability chapter. There
the missingness is specific to the measure rather than to the reply: a reply
below the length floor still carries its vocabulary and structure measures and
is undefined only on the five readability formulas, and a reply whose words are
absent from the age of acquisition norms is undefined only on the four vocabulary
measures. So there is no single readability denominator to add here, and each is
established alongside the measure it belongs to.


In [4]:
flow = pd.DataFrame({
    'Submitted': frame.groupby('label').size(),
    'Blocked': frame.groupby('label')['blocked'].sum(),
}).reindex(ORDER).astype('Int64')
flow.loc['Total'] = flow.sum()
flow['Block Rate (%)'] = (flow['Blocked'] / flow['Submitted'] * 100).round(2)
flow.index.name = 'Model'

publish(flow.reset_index(), 'corpus_01_flow')


,Model,Submitted,Blocked,Block Rate (%)
0,GPT-5.6 Luna,7800,0,0.0
1,Claude Haiku 4.5,7800,1,0.01
2,Gemini 3.5 Flash Lite,7800,159,2.04
3,DeepSeek-V4 Flash,7800,0,0.0
4,Mistral Small 4,7800,0,0.0
5,Gemma 4 31B,7800,0,0.0
6,Total,46800,160,0.34


In [5]:
# Eligible against analysed, rather than a share of the whole corpus. How much of
# the corpus a stratum represents is a design allocation and says nothing about
# data loss; what a reader needs is how much of each eligible subset survived to
# be analysed, which here is entirely a question of blocking.
everything = pd.Series(True, index=frame.index)
restricted = frame['scenario_type'].eq('Age Restricted')

SUBSETS = [
    ('Provider Block Rate', 'All Submitted Requests', everything),
    ('Refusal Rate, Content Delivery Rate', 'All Returned Replies', everything),
    ('Benchmark Action Alignment', 'Replies With an Expected Answer',
     carries(frame['expected_answer'])),
    ('Harmful Content Delivered', 'Harmful Scenarios',
     frame['scenario_type'].eq('Harmful')),
    ('Age Restricted Content Delivered',
     'Age Restricted, Explicit Age (Minor)',
     restricted & frame['condition'].isin(STATED_MINOR)),
    ('Age Restricted Request Refused',
     'Age Restricted, Explicit Age (Adult)',
     restricted & frame['condition'].isin(STATED_ADULT)),
    ('Rights and Benign Over-Refusal', 'Rights or Benign Scenarios',
     frame['scenario_type'].isin(['Rights', 'Benign'])),
]

rows = []
for name, subset, mask in SUBSETS:
    eligible = int(mask.sum())
    analysed = (eligible if name.startswith('Provider')
                else int(frame.loc[mask, 'responded'].sum()))
    rows.append({'Rate': name, 'Eligible subset': subset,
                 'Eligible submitted': eligible,
                 'Analysis denominator': analysed,
                 'Available (%)': round(analysed / eligible * 100, 2)})

denominators = pd.DataFrame(rows)
publish(denominators, 'corpus_s01_denominators')


,Rate,Eligible Subset,Eligible Submitted,Analysis Denominator,Available (%)
0,Provider Block Rate,All Submitted Requests,46800,46800,100.00
1,"Refusal Rate, Content Delivery Rate",All Returned Replies,46800,46640,99.66
2,Benchmark Action Alignment,Replies With an Expected Answer,46350,46190,99.65
3,Harmful Content Delivered,Harmful Scenarios,11700,11658,99.64
4,Age Restricted Content Delivered,"Age Restricted, Explicit Age (Minor)",2700,2621,97.07
5,Age Restricted Request Refused,"Age Restricted, Explicit Age (Adult)",900,900,100.00
6,Rights and Benign Over-Refusal,Rights or Benign Scenarios,29250,29211,99.87


## C.4 Provider blocking as a platform outcome

A block is a provider-blocked request for which no response text was returned.
Nothing was said and nothing can be classified. Treating it only as missingness would discard whatever
the block itself carries, so it is analysed here as an outcome in its own right
and then excluded from the safety denominator, where it does not belong: no model
behaviour occurred.

What can be claimed from this is an association. Provider blocks were
overwhelmingly concentrated in conditions in which a minor age was stated
explicitly, with two blocked requests under one implicit minor condition, and
and no provider block was observed at ages eighteen or twenty-one.
The pipeline that produced that pattern is not observable from the outside, so
the finding is that provider blocking is associated with explicit minor-age
disclosure, not that an age-gating mechanism has been identified.

Two contrasts, declared in the `platform blocking` family, secondary tier. Both
mirror contrasts the safety chapter runs on returned responses, which is what
makes the two layers comparable: block rate under stated minor ages against
stated adult ages, and block rate under stated minor ages against implicit minor
cues. Only one model blocked enough to test; the other is reported as a count.


In [6]:
blocking = frame[frame['blocked'] > 0]
by_model = frame.groupby('label')['blocked'].sum().reindex(ORDER)
model = by_model.idxmax()


In [7]:
# One model carries effectively all of the blocking, so the condition by domain
# grid is built for it. Only the domains in which that model blocked at least
# once are shown; the other seven would be a column of noughts.
panel = frame[frame['label'] == model]
hot = [domain for domain in frame['domain'].dropna().drop_duplicates()
       if domain in set(blocking.loc[blocking['label'] == model, 'domain'])]

panel = panel.assign(Condition=panel['condition'].map(CONDITION_LABEL))
grid = (panel.pivot_table(index='Condition', columns='domain',
                          values='blocked', aggfunc='mean') * 100)
grid = grid.reindex(columns=hot)
grid.insert(0, 'All Domains',
            panel.groupby('Condition')['blocked'].mean() * 100)
grid = grid.reindex(LABEL_ORDER).round(1)

publish(grid.reset_index(), 'corpus_02_blocking')


domain,Condition,All Domains,Violence,Sexual Content,Bullying
0,Control (No Age),0.0,0.0,0.0,0.0
1,Explicit Age (7),5.0,5.0,45.0,0.0
2,Explicit Age (9),5.5,5.0,50.0,0.0
3,Explicit Age (11),6.0,5.0,55.0,0.0
4,Explicit Age (13),4.5,5.0,40.0,0.0
5,Explicit Age (15),4.2,5.0,36.7,0.0
6,Explicit Age (17),1.2,0.0,11.7,0.0
7,Explicit Age (18),0.0,0.0,0.0,0.0
8,Explicit Age (21),0.0,0.0,0.0,0.0
9,Implicit Cue (Minor),0.1,0.0,0.0,0.8


In [8]:
CONTRASTS = [(TRAJECTORY, STATED_MINOR, STATED_ADULT),
             (SIGNAL, STATED_MINOR, IMPLICIT_MINOR)]

# Every model enters the family, including those that blocked nothing, so that
# admission does not depend on the outcome being tested.
for name, first, second in CONTRASTS:
    contrast(frame, 'blocked', first, second, 'platform blocking', name,
             REGISTER)

# The platform blocking family is complete at this point and no other notebook
# writes into it, so it can be corrected here. Everything below reads the
# adjusted register rather than the raw values.
blocked_register = analysis.adjust(REGISTER.frame())
# All twelve are registered and corrected. Ten of them are the same row of
# zeros, so the table shows the model that moved, one line standing for the
# five that did not, and the panel average.
rows = []
for name, _, _ in CONTRASTS:
    part = present(blocked_register, name).set_index('Model')
    quiet = [m for m in ORDER if m != model]
    keep = ['Effect (pp)', 'p', 'q', '95% CI Lower', '95% CI Upper']
    rows.append({'Contrast': name, 'Model': model,
                 **part.loc[model, keep].to_dict()})
    span = sorted(float(part.loc[m, 'Effect (pp)']) for m in quiet)
    rows.append({'Contrast': name, 'Model': 'Five other models',
                 'Effect (pp)': (f'{span[0]:+.1f}' if span[0] == span[-1]
                                 else f'{span[0]:+.1f} to {span[-1]:+.1f}'),
                 'p': '1.000', 'q': '1.000'})
    rows.append({'Contrast': name, 'Model': MACRO,
                 **part.loc[MACRO, keep].to_dict()})
tested = [pd.DataFrame(rows)]
blocking_tests = pd.concat(tested).reset_index(drop=True)
publish(blocking_tests, 'corpus_s02_blocking_tests')


,Contrast,Model,Effect (pp),p,q,95% CI Lower,95% CI Upper
0,Explicit Age (Minor) vs Explicit Age (Adult),Gemini 3.5 Flash Lite,+4.4,< 0.001,0.004,2.1,7.1
1,Explicit Age (Minor) vs Explicit Age (Adult),Five other models,+0.0,1.000,1.000,NaN,NaN
2,Explicit Age (Minor) vs Explicit Age (Adult),Macro-average,+0.7,,,0.3,1.2
3,Explicit Age (Minor) vs Implicit Cue (Minor),Gemini 3.5 Flash Lite,+4.3,< 0.001,0.004,1.9,7.1
4,Explicit Age (Minor) vs Implicit Cue (Minor),Five other models,-0.1 to +0.0,1.000,1.000,NaN,NaN
5,Explicit Age (Minor) vs Implicit Cue (Minor),Macro-average,+0.7,,,0.3,1.2


## C.5 Within-cell variation

Three replicates were collected at the same sampling setting, so a cell whose
replicates differ reflects stochastic variation under that fixed setting rather
than a difference in policy. That is worth a number: it characterises how much
stochastic variability there is at the fixed sampling setting, and it is a caveat
that attaches to a model rather than to a finding. It does not bound the size any
later effect could take, and is not offered as though it did.

Variation is computed only on cells where all three replicates returned a
response. A provider-blocked request carries no outcome cell at all, so a cell
that was blocked on every replicate would otherwise count as invariant on a value
none of its replicates has. Cells that lost some or all replicates to blocking
are reported separately, beside a provider-status consistency that asks whether
the blocking itself was consistent.

The outcome is Main Response crossed with Delivery Response, so it varies exactly
when one of the two does. That gives three disjoint regions, reported as three
columns: cells where only Main Response varied, cells where only Delivery
Response varied, and cells where both did. They sum to the outcome column, and
each is counted directly rather than obtained by subtracting two rounded rates.

Replicate-level variability is analysed only here. Everywhere else the three
replicates are averaged within each scenario x condition cell before inference,
so they never enlarge a sample.


In [9]:
grouped = frame.groupby(['label', 'scenario_id', 'prompt_id'])
per_cell = pd.DataFrame({
    'returned': grouped['responded'].sum(),
    'blocked': grouped['blocked'].sum(),
    'answers': grouped['answer'].apply(
        lambda values: values[values.ne(analysis.BLOCKED)].nunique()),
    'deliveries': grouped[analysis.DELIVERY].apply(
        lambda values: values.dropna().nunique()),
}).reset_index()
# A cell is consistent on provider status when every replicate went the same way,
# all returned or all blocked. It is a different question from the outcome and is
# defined on every cell.
per_cell['status_consistent'] = per_cell['blocked'].isin(
    [0, analysis.DESIGN['replicates']]).astype(float)

whole = per_cell[per_cell['returned'] == analysis.DESIGN['replicates']].copy()
# The outcome is Main Response crossed with Delivery Response, so it varies
# exactly when one of them does, and the three regions below are disjoint and sum
# to it. Each is counted on the cells rather than obtained by subtracting two
# rounded percentages, which is what the earlier gap column did.
moved_main = whole['answers'].gt(1)
moved_delivery = whole['deliveries'].gt(1)
whole['main_only'] = (moved_main & ~moved_delivery).astype(float)
whole['delivery_only'] = (~moved_main & moved_delivery).astype(float)
whole['overlap'] = (moved_main & moved_delivery).astype(float)
whole['outcome_varied'] = (moved_main | moved_delivery).astype(float)
assert (whole['main_only'] + whole['delivery_only'] + whole['overlap']
        == whole['outcome_varied']).all(), 'the three regions are not disjoint'

rows, points = {}, {}
for label in ORDER:
    part = whole[whole['label'] == label]
    outcome = bootstrap_rate(part, 'outcome_varied')
    points[label] = outcome[0] * 100
    split = analysis.bounds(*[value * 100 for value in outcome])
    rows[label] = {
        'Main Response (%)': round(part['main_only'].mean() * 100, 1),
        'Delivery Response (%)': round(part['delivery_only'].mean() * 100, 1),
        'Overlap (%)': round(part['overlap'].mean() * 100, 1),
        'Outcome (%)': split['estimate'],
        '95% CI Lower': split['low'],
        '95% CI Upper': split['high'],
    }

variation = (pd.DataFrame(rows).T.reindex(ORDER)
             .rename_axis('Model').reset_index())

# The leave-one-model-out figures are six numbers carrying one fact, so they are
# stated rather than given a column of their own.
without = leave_one_out(pd.Series(points)).round(1)
others = without.drop('Mistral Small 4', errors='ignore')
print(f'macro-average outcome variation {np.mean(list(points.values())):.1f} '
      f'per cent; without Mistral Small 4 {without["Mistral Small 4"]:.1f}, '
      f'without any other model between {others.min():.1f} and {others.max():.1f}')

publish(variation, 'corpus_03_unanimity')


macro-average outcome variation 12.1 per cent; without Mistral Small 4 7.0, without any other model between 12.7 and 13.5


,Model,Main Response (%),Delivery Response (%),Overlap (%),Outcome (%),95% CI Lower,95% CI Upper
0,GPT-5.6 Luna,1.3,2.5,1.9,5.7,3.8,7.7
1,Claude Haiku 4.5,3.0,3.8,1.8,8.6,6.3,11.1
2,Gemini 3.5 Flash Lite,1.6,2.4,3.1,7.2,5.0,9.5
3,DeepSeek-V4 Flash,3.3,1.5,4.0,8.8,6.5,11.3
4,Mistral Small 4,7.8,22.3,7.1,37.2,33.5,40.9
5,Gemma 4 31B,1.2,1.7,2.1,5.0,3.5,6.6


In [10]:
# Only the two models that lost cells. The other four are zero on every column
# and are stated in the caption rather than given four identical rows.
excluded = pd.DataFrame({
    'No replicate returned': per_cell.groupby('label')['returned']
                             .apply(lambda v: int(v.eq(0).sum())),
    'Some replicates returned': per_cell.groupby('label')['returned']
                                .apply(lambda v: int(v.between(1, 2).sum())),
    'Provider-status unanimity (%)': (per_cell.groupby('label')['status_consistent']
                                      .mean() * 100).round(2),
}).reindex(ORDER)
excluded = excluded[excluded[['No replicate returned',
                             'Some replicates returned']].sum(axis=1) > 0]
excluded.index.name = 'Model'

publish(excluded, 'corpus_s03_unanimity_excluded')


,No Replicate Returned,Some Replicates Returned,Provider-Status Unanimity (%)
Model,,,
Claude Haiku 4.5,0,1,99.96
Gemini 3.5 Flash Lite,51,5,99.81


## C.6 Register and Outputs

The one family this notebook populates, `platform blocking`, is complete here,
so Benjamini and Hochberg was applied to it in C.4 and the values shown there
are final. The register is written to `tables/machine/register_corpus.csv` so
that the reporting notebook can assemble the audit table across every notebook.


In [11]:
register = REGISTER.frame()
REGISTER.write()
write_captions()
print(f'{len(register)} tests registered; '
      + ', '.join(f'{count} {kind}' for kind, count
                  in sorted(analysis.WRITTEN.items())) + ' written')
register


described in captions.yml but not written: corpus_04_reliability, corpus_s04_reliability_full, corpus_s05_repeatability
14 tests registered; 1 main table, 1 methods table, 4 supplement table written


,prefix,tier,family,contrast,measure,model,n,effect,low,high,p
0,corpus,secondary,platform blocking,Explicit Age (Minor) vs Explicit Age (Adult),blocked,GPT-5.6 Luna,200,0.000000,0.000000,0.000000,1.000000
1,corpus,secondary,platform blocking,Explicit Age (Minor) vs Explicit Age (Adult),blocked,Claude Haiku 4.5,200,0.000000,0.000000,0.000000,1.000000
2,corpus,secondary,platform blocking,Explicit Age (Minor) vs Explicit Age (Adult),blocked,Gemini 3.5 Flash Lite,200,4.388889,2.054861,7.138889,0.000488
3,corpus,secondary,platform blocking,Explicit Age (Minor) vs Explicit Age (Adult),blocked,DeepSeek-V4 Flash,200,0.000000,0.000000,0.000000,1.000000
4,corpus,secondary,platform blocking,Explicit Age (Minor) vs Explicit Age (Adult),blocked,Mistral Small 4,200,0.000000,0.000000,0.000000,1.000000
5,corpus,secondary,platform blocking,Explicit Age (Minor) vs Explicit Age (Adult),blocked,Gemma 4 31B,200,0.000000,0.000000,0.000000,1.000000
6,corpus,secondary,platform blocking,Explicit Age (Minor) vs Explicit Age (Adult),blocked,Macro-average,<NA>,0.731481,0.342477,1.189815,NaN
7,corpus,secondary,platform blocking,Explicit Age (Minor) vs Implicit Cue (Minor),blocked,GPT-5.6 Luna,200,0.000000,0.000000,0.000000,1.000000
8,corpus,secondary,platform blocking,Explicit Age (Minor) vs Implicit Cue (Minor),blocked,Claude Haiku 4.5,200,-0.083333,-0.250000,0.000000,1.000000
9,corpus,secondary,platform blocking,Explicit Age (Minor) vs Implicit Cue (Minor),blocked,Gemini 3.5 Flash Lite,200,4.305556,1.944444,7.055556,0.000732
